# ETF Document Loader

Load the full ETF universe from FinanceDatabase into ChromaDB.
Data is sourced from the FinanceDatabase package (~36K ETFs, updated weekly).
No Tavily enrichment needed — the `summary` field from fund prospectuses is used directly.

In [1]:
%reload_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))

from apps.agentic.core.document_loaders.etf.finance_database_loader import FinanceDatabaseLoader
from apps.agentic.core.agents.chroma_rag_agent import _normalize_chroma_filter
from apps.agentic.core.constants import ETF_DB_NAME, ETF_COLLECTION_NAME
from apps.agentic.core.utils import set_chatgpt_env

DB_PATH = Path("../../.db").resolve()
DB_PATH.mkdir(parents=True, exist_ok=True)

set_chatgpt_env(filedir="../../.keys")

## Configuration

In [2]:
loader = FinanceDatabaseLoader(db_path=str(DB_PATH))
ETF_DB_NAME, ETF_COLLECTION_NAME, DB_PATH

('etf', 'etf-info', PosixPath('/Users/troy/Develop/gly.fish/yada/.db'))

## Delete all documents in the database

In [7]:
# loader.delete_all()

INFO:     FinanceDatabaseLoader: deleted 109811 documents from 'etf-info'


109811

## Count Documents

In [8]:
coll = loader.vectorstore._collection
print("Collection:", coll.name)
print("Docs:", coll.count())

Collection: etf-info
Docs: 36475


## Load ETFs

In [4]:
# Test load — VanEck only (~386 funds, ~1 minute)
# For the full 36K universe: await loader.load_all_documents()

# await loader.load_all_documents(family="VanEck Asset Management")
# print("Docs after load:", coll.count())

INFO:     FinanceDatabaseLoader: downloading ETF data from FinanceDatabase...
INFO:     FinanceDatabaseLoader: 386 ETFs downloaded, building documents...
INFO:     FinanceDatabaseLoader: 386 documents built (0 skipped), writing to ChromaDB...
INFO:     FinanceDatabaseLoader: 386/386 documents written...
INFO:     FinanceDatabaseLoader: done — 386 ETF documents in collection 'etf-info'.
Docs after load: 36861


In [23]:
## Inspect stored documents
probe = coll.get(limit=5)
docs: list = probe.get("documents") or []
metas: list = probe.get("metadatas") or []
for i, (doc, meta) in enumerate(zip(docs, metas)):
    print(f"\n--- [{i}] {meta.get('ticker')} | {meta.get('family')} ---")
    print(doc[:400])


--- [0] 003D.BE | WisdomTree Asset Management ---
# WISDOMTREE EURO HGD EQ.FD
- **Ticker:** 003D.BE
- **Fund Family:** WisdomTree Asset Management
- **Asset Class:** Financials
- **Category:** Developed Markets
- **Exchange:** BER
- **Currency:** EUR

## Description
The WisdomTree Euro Hedged Equity Fund aims to provide exposure to European equities while hedging against fluctuations between the euro and the U.S. dollar. By hedging the currency r

--- [1] 003H.BE | WisdomTree Asset Management ---
# WISDOMTREE EM.CURR.STRAT.
- **Ticker:** 003H.BE
- **Fund Family:** WisdomTree Asset Management
- **Asset Class:** Currencies
- **Category:** Emerging Markets
- **Exchange:** BER
- **Currency:** EUR

## Description
The WisdomTree Emerging Markets Currency Strategy Fund seeks to provide exposure to emerging market currencies. The fund uses a rules-based methodology to select and weight currencies 

--- [2] 003I.BE | WisdomTree Asset Management ---
# WISDOMTREE INTL H.Q.D.GR.
- **Ticker:** 003

## Metadata Key Values

Scan the full collection to find all distinct values for each filterable metadata field.
Use these to validate filter values before passing them to the agent or similarity search.

In [38]:
FILTER_KEYS = ["family", "category_group", "category", "exchange", "currency"]

total = coll.count()
distinct: dict[str, set] = {k: set() for k in FILTER_KEYS}

BATCH = 5000
for offset in range(0, total, BATCH):
    result = coll.get(limit=BATCH, offset=offset, include=["metadatas"])
    for meta in result.get("metadatas") or []:
        for key in FILTER_KEYS:
            val = meta.get(key)
            if val:
                distinct[key].add(val)

for key in FILTER_KEYS:
    values = sorted(distinct[key])
    print(f"\n{key} ({len(values)} distinct values):")
    for v in values:
        print(f"  {v!r}")


family (317 distinct values):
  '1832 Asset Management'
  '21Shares'
  '2nd Vote Funds'
  '3 Banken-Generali Investment-Gesellschaft'
  'ACATIS Investment Management'
  'ACV ETFs'
  'AGF Investments'
  'ALPS ETF Trust'
  'ARK ETF Trust'
  'ASYMshares'
  'ATAC Fund'
  'AXA Investment Management'
  'AXA Investment Managers'
  'AXA World Funds'
  'Aberdeen Standard Investments'
  'Absolute Investment Advisers'
  'Accelerate Financial Technologies'
  'Acquirers Funds'
  'Acruence'
  'Actinver Casa de Bolsa'
  'Adasina'
  'AdvisorShares'
  'Aegon Investment Management'
  'Affinity'
  'Alerian'
  'Alger'
  'AllianceBernstein'
  'Allianz Global Investors Fund'
  'Allianz Investment Management'
  'Alpha Architect'
  'AlphaClone'
  'AlphaMark'
  'AltShares'
  'Alternative Access Funds'
  'American Century Investments'
  'Amphitryon Capital Management'
  'Amplify ETFs'
  'Amundi Asset Management'
  'Anfield'
  'Aptus Capital Advisors'
  'ArrowShares'
  'Asset Management One'
  'Avantis Investor

In [40]:
# Search for a specific value within a metadata key — useful for checking
# what the LLM extractor would need to produce to match stored data.
search_key = "family"
search_term = "VanEck"

matches = sorted(v for v in distinct[search_key] if search_term.lower() in v.lower())
print(f"{search_key} values containing {search_term!r}:")
for v in matches:
    print(f"  {v!r}")

family values containing 'VanEck':
  'VanEck Asset Management'


In [39]:
## Similarity search — unfiltered
results = loader.vectorstore.similarity_search("international equity dividend growth", k=5)
for r in results:
    print(r.metadata.get("ticker"), "|", r.metadata.get("name"), "|", r.metadata.get("family"))
    print(r.page_content[:200])
    print()

F4FC.BE | ISHSTR-INTL.DIVID.GROWTH | BlackRock Asset Management
# ISHSTR-INTL.DIVID.GROWTH
- **Ticker:** F4FC.BE
- **Fund Family:** BlackRock Asset Management
- **Asset Class:** Alternatives
- **Category:** Developed Markets
- **Exchange:** BER
- **Currency:** EUR

DJFG.F | Allianz Global Equity Dividend | Allianz Global Investors Fund
# Allianz Global Equity Dividend
- **Ticker:** DJFG.F
- **Fund Family:** Allianz Global Investors Fund
- **Exchange:** FRA
- **Currency:** EUR

## Description
Allianz Global Equity Dividend focuses on

IDV.MX | iShares International Select Dividend ETF | BlackRock Asset Management
# iShares International Select Dividend ETF
- **Ticker:** IDV.MX
- **Fund Family:** BlackRock Asset Management
- **Asset Class:** Financials
- **Category:** Developed Markets
- **Exchange:** MEX
- **C

IDV | iShares International Select Dividend ETF | BlackRock Asset Management
# iShares International Select Dividend ETF
- **Ticker:** IDV
- **Fund Family:** BlackRock Asset Mana

In [12]:
## Similarity search — filtered
# ChromaDB 1.0.x requires exactly one top-level operator.
# Use _normalize_chroma_filter to wrap multi-key dicts in $and automatically.

raw_filter = {
    "family": "VanEck Asset Management",
    "exchange": {"$in": ["ASE", "NCM", "NIM", "NYQ", "PCX", "PNK"]},
    # "category_group": "Equities",
    # "category": "Dividend Growth",
}
chroma_filter = _normalize_chroma_filter(raw_filter)
print("Filter:", chroma_filter)

results = loader.vectorstore.similarity_search(
    "List all funds",
    k=500,
    filter=chroma_filter,
)
print(f"\nResults for query: {len(results)}")
for r in results:
    print(r.metadata.get("ticker"), "|", r.metadata.get("name"))
    print(r.page_content[:200])
    print()

Filter: {'$and': [{'family': 'VanEck Asset Management'}, {'exchange': {'$in': ['ASE', 'NCM', 'NIM', 'NYQ', 'PCX', 'PNK']}}]}

Results for query: 35
AFK | VanEck Vectors Africa Index ETF
# VanEck Vectors Africa Index ETF
- **Ticker:** AFK
- **Fund Family:** VanEck Asset Management
- **Asset Class:** Equities
- **Category:** Frontier Markets
- **Exchange:** PCX
- **Currency:** USD

## 

EMAG | VanEck Vectors Emerging Markets Aggregate Bond ETF
# VanEck Vectors Emerging Markets Aggregate Bond ETF
- **Ticker:** EMAG
- **Fund Family:** VanEck Asset Management
- **Asset Class:** Fixed Income
- **Category:** Corporate Bonds
- **Exchange:** PCX
-

PFXF | VanEck Vectors Preferred Securities ex Financials ETF
# VanEck Vectors Preferred Securities ex Financials ETF
- **Ticker:** PFXF
- **Fund Family:** VanEck Asset Management
- **Asset Class:** Financials
- **Category:** Corporate Bonds
- **Exchange:** PCX


RAAX | VanEck Vectors Real Asset Allocation ETF
# VanEck Vectors Real Asset Allocation ET